# LLM 推理策略综合指南

本教程将所有推理策略融合贯通，帮助你理解它们的关系、选择和组合使用。

## 目录
1. [推理策略全景图](#1-推理策略全景图)
2. [策略对比与选择](#2-策略对比与选择)
3. [策略组合模式](#3-策略组合模式)
4. [完整实战案例](#4-完整实战案例)
5. [决策流程图](#5-决策流程图)

In [ ]:
import sys
sys.path.insert(0, '..')

# 导入所有模块
from src.chain_of_thought import ZeroShotCoT, FewShotCoT, AutoCoT, CoTExample
from src.react import ReActAgent, SimpleTool, ReActTrace
from src.tree_of_thoughts import TreeOfThoughts, BeamSearch, ThoughtNode
from src.self_consistency import SelfConsistency, MajorityVoting, SampledPath
from src.reflection import Reflection, SimpleSelfEvaluator

print("所有模块加载成功！")

---
## 1. 推理策略全景图

### 1.1 五大策略概览

In [ ]:
print("""
┌─────────────────────────────────────────────────────────────────────┐
│                    LLM 推理策略全景图                                │
├─────────────────────────────────────────────────────────────────────┤
│                                                                     │
│  ┌─────────────┐    ┌─────────────┐    ┌─────────────┐             │
│  │    CoT      │    │   ReAct     │    │    ToT      │             │
│  │  思维链     │    │ 推理+行动   │    │   思维树    │             │
│  │  ─────────  │    │  ─────────  │    │  ─────────  │             │
│  │ 线性推理    │    │ 工具调用    │    │ 树搜索      │             │
│  │ 步骤分解    │    │ 环境交互    │    │ 多路径探索  │             │
│  └─────────────┘    └─────────────┘    └─────────────┘             │
│         │                  │                  │                    │
│         └──────────────────┼──────────────────┘                    │
│                            │                                       │
│                            ▼                                       │
│  ┌─────────────────────────────────────────────────┐               │
│  │              增强策略 (可与上述组合)              │               │
│  │  ┌───────────────────┐  ┌───────────────────┐  │               │
│  │  │ Self-Consistency  │  │    Reflection     │  │               │
│  │  │    多次采样投票    │  │    迭代自我改进    │  │               │
│  │  └───────────────────┘  └───────────────────┘  │               │
│  └─────────────────────────────────────────────────┘               │
│                                                                     │
└─────────────────────────────────────────────────────────────────────┘
""")

### 1.2 策略演进关系

In [ ]:
print("""
策略演进关系：

基础 ──────────────────────────────────────────────────> 高级

直接回答 → Zero-shot CoT → Few-shot CoT → ReAct → ToT
   │              │              │           │       │
   │              │              │           │       │
   └──────────────┴──────────────┴───────────┴───────┘
                          │
                          ▼
              + Self-Consistency (多采样)
              + Reflection (迭代改进)

复杂度: 低 ────────────────────────────────────────────> 高
效果:   基础 ──────────────────────────────────────────> 最优
成本:   低 ────────────────────────────────────────────> 高
""")

---
## 2. 策略对比与选择

### 2.1 详细对比表

In [ ]:
print("""
┌────────────────┬──────────┬──────────┬──────────┬──────────┬──────────┐
│      特性      │   CoT    │  ReAct   │   ToT    │   SC     │ Reflect  │
├────────────────┼──────────┼──────────┼──────────┼──────────┼──────────┤
│ 推理结构       │ 线性     │ 循环     │ 树形     │ 并行     │ 迭代     │
│ 外部工具       │ ✗        │ ✓        │ ✗        │ ✗        │ ✗        │
│ 多路径探索     │ ✗        │ ✗        │ ✓        │ ✓        │ ✗        │
│ 自我纠错       │ ✗        │ 部分     │ 部分     │ ✗        │ ✓        │
│ API调用次数    │ 1        │ 多次     │ 多次     │ N次      │ 2-4次    │
│ 实现复杂度     │ 低       │ 中       │ 高       │ 低       │ 中       │
│ 适用模型大小   │ >100B    │ >10B     │ >100B    │ >100B    │ >10B     │
└────────────────┴──────────┴──────────┴──────────┴──────────┴──────────┘

SC = Self-Consistency, Reflect = Reflection
""")

### 2.2 场景选择指南

In [ ]:
def recommend_strategy(task_type: str, needs_tools: bool, 
                       complexity: str, accuracy_required: str) -> list:
    """根据任务特征推荐策略"""
    recommendations = []
    
    # 基础策略选择
    if needs_tools:
        recommendations.append("ReAct (需要工具调用)")
    elif complexity == "high":
        recommendations.append("ToT (复杂问题探索)")
    elif complexity == "medium":
        recommendations.append("Few-shot CoT (中等复杂度)")
    else:
        recommendations.append("Zero-shot CoT (简单任务)")
    
    # 增强策略
    if accuracy_required == "high":
        recommendations.append("+ Self-Consistency (提高可靠性)")
    if task_type in ["writing", "code"]:
        recommendations.append("+ Reflection (迭代改进)")
    
    return recommendations

# 测试不同场景
scenarios = [
    ("math", False, "medium", "high"),
    ("search", True, "low", "medium"),
    ("writing", False, "high", "high"),
    ("code", False, "medium", "high"),
]

print("场景推荐：")
for task, tools, comp, acc in scenarios:
    recs = recommend_strategy(task, tools, comp, acc)
    print(f"\n  任务={task}, 工具={tools}, 复杂度={comp}, 精度={acc}")
    for r in recs:
        print(f"    → {r}")

---
## 3. 策略组合模式

### 3.1 常见组合

In [ ]:
print("""
常见策略组合模式：

1. CoT + Self-Consistency (最常用)
   ┌─────────────────────────────────────┐
   │ 问题 → [CoT采样1] → 答案A          │
   │      → [CoT采样2] → 答案A          │
   │      → [CoT采样3] → 答案B          │
   │      → [CoT采样4] → 答案A          │
   │                     ↓              │
   │              投票 → 答案A          │
   └─────────────────────────────────────┘

2. ReAct + Reflection (Agent增强)
   ┌─────────────────────────────────────┐
   │ ReAct执行 → 结果 → 自我评估        │
   │                      ↓              │
   │              不满意 → 重新执行      │
   │              满意   → 返回结果      │
   └─────────────────────────────────────┘

3. ToT + Self-Consistency (高精度)
   ┌─────────────────────────────────────┐
   │ ToT搜索 → 多个候选解               │
   │              ↓                      │
   │         投票选择最优解              │
   └─────────────────────────────────────┘
""")

### 3.2 组合实现示例

In [ ]:
# CoT + Self-Consistency 组合
def cot_with_consistency(question: str, n_samples: int = 5):
    """CoT + Self-Consistency 组合策略"""
    cot = FewShotCoT(examples=[
        CoTExample("5+3=?", "5+3=8", "8"),
        CoTExample("10-4=?", "10-4=6", "6"),
    ])
    
    # 模拟多次采样
    paths = []
    for i in range(n_samples):
        prompt = cot.get_prompt(question)
        # 实际应用中这里调用 LLM
        paths.append(SampledPath(
            reasoning=f"推理路径{i+1}",
            answer="12",  # 模拟答案
            confidence=0.8
        ))
    
    # 投票
    voting = MajorityVoting()
    answer, conf = voting.vote(paths)
    return answer, conf

answer, conf = cot_with_consistency("15%的80是多少？")
print(f"CoT + SC 结果: {answer} (置信度: {conf:.0%})")

In [ ]:
# ReAct + Reflection 组合
def react_with_reflection(task: str, max_retries: int = 3):
    """ReAct + Reflection 组合策略"""
    tools = [
        SimpleTool("calculator", "计算", lambda x: str(eval(x))),
        SimpleTool("finish", "完成", lambda x: x),
    ]
    agent = ReActAgent(tools=tools, max_iterations=5)
    reflection = Reflection(max_iterations=2, target_score=0.8)
    
    for attempt in range(max_retries):
        # ReAct 执行
        result = agent.run(task)
        
        # Reflection 评估
        ref_result = reflection.reflect(task, str(result))
        
        if ref_result.improved:
            return ref_result.final_response
    
    return result

print("ReAct + Reflection 组合已定义")

---
## 4. 完整实战案例

### 4.1 案例：复杂数学问题

In [ ]:
print("""
案例：解决复杂数学问题
问题：一个班有30人，男生比女生多6人，男生有多少人？

策略选择：Few-shot CoT + Self-Consistency

执行过程：
─────────────────────────────────────────────────────
采样1: 设女生x，男生x+6，2x+6=30，x=12，男生=18 ✓
采样2: (30+6)/2 = 18 ✓
采样3: 30/2 + 3 = 18 ✓
采样4: 30-6=24, 24/2=12 ✗ (计算错误)
采样5: 男生=(30+6)/2=18 ✓
─────────────────────────────────────────────────────
投票结果: 18(4票) vs 12(1票)
最终答案: 18人
置信度: 80%
""")

### 4.2 案例：信息检索任务

In [ ]:
print("""
案例：查询并计算
问题：查询北京人口，计算人均GDP（GDP=4万亿）

策略选择：ReAct

执行过程：
─────────────────────────────────────────────────────
Thought: 需要先查询北京人口
Action: search(北京人口)
Observation: 北京常住人口约2200万

Thought: 现在计算人均GDP
Action: calculator(40000/2200)
Observation: 18.18

Thought: 已得到结果
Action: finish(北京人均GDP约18.18万元)
─────────────────────────────────────────────────────
最终答案: 北京人均GDP约18.18万元
""")

### 4.3 案例：创意写作

In [ ]:
print("""
案例：写一首关于AI的诗

策略选择：ToT + Reflection

执行过程：
─────────────────────────────────────────────────────
ToT 探索：
        [AI诗歌]
       /    |    \\
   [科幻]  [哲学]  [抒情]
   0.6     0.8     0.7
            ↓
      选择[哲学]风格

Reflection 改进：
  迭代1: 初稿 → 评估(0.6) → 改进韵律
  迭代2: 修改稿 → 评估(0.85) → 通过
─────────────────────────────────────────────────────
最终输出: 经过优化的哲学风格AI诗歌
""")

---
## 5. 决策流程图

In [ ]:
print("""
推理策略决策流程：

                    ┌─────────────┐
                    │   开始      │
                    └──────┬──────┘
                           │
                           ▼
                ┌──────────────────────┐
                │  需要调用外部工具？   │
                └──────────┬───────────┘
                     是 /     \\ 否
                       /       \\
                      ▼         ▼
               ┌─────────┐  ┌──────────────────┐
               │  ReAct  │  │  问题复杂度？     │
               └─────────┘  └────────┬─────────┘
                                高 / │ \\ 低
                                  /  │  \\
                                 ▼   │   ▼
                           ┌─────┐   │  ┌──────────┐
                           │ ToT │   │  │Zero-shot │
                           └─────┘   │  │   CoT    │
                                     │  └──────────┘
                                     ▼
                              ┌───────────┐
                              │ Few-shot  │
                              │    CoT    │
                              └───────────┘
                                     │
                                     ▼
                        ┌────────────────────────┐
                        │  需要高可靠性？         │
                        └───────────┬────────────┘
                              是 /     \\ 否
                                /       \\
                               ▼         ▼
                    ┌──────────────────┐  │
                    │+ Self-Consistency│  │
                    └──────────────────┘  │
                                          │
                                          ▼
                        ┌────────────────────────┐
                        │  需要迭代改进？         │
                        └───────────┬────────────┘
                              是 /     \\ 否
                                /       \\
                               ▼         ▼
                    ┌──────────────────┐  ┌────────┐
                    │  + Reflection    │  │  完成  │
                    └──────────────────┘  └────────┘
""")

---
## 总结

### 核心要点

1. **CoT** - 基础推理，分步思考
2. **ReAct** - 需要工具时使用
3. **ToT** - 复杂问题多路径探索
4. **Self-Consistency** - 提高可靠性
5. **Reflection** - 迭代改进质量

### 组合建议

- 数学/逻辑: CoT + Self-Consistency
- 信息检索: ReAct
- 创意任务: ToT + Reflection
- 代码生成: CoT + Reflection
- 高精度要求: 任意策略 + Self-Consistency